<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Finance (2nd ed.)

**Mastering Data-Driven Finance**

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

<img src="https://hilpisch.com/images/py4fi_2nd_shadow.png" width="300px" align="left">

<b style="color: red; font-size: 16px;">FXCM has stopped the original API support.<br>Therefore the code in this notebook does unfortunately not work anymore.</b>

# Trading Platform

## Risk Disclaimer

<font size="-1">
Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits. Leverage can work against you. Due to the certain restrictions imposed by the local law and regulation, German resident retail client(s) could sustain a total loss of deposited funds but are not subject to subsequent payment obligations beyond the deposited funds. Be aware and fully understand all risks associated with the market and trading. Prior to trading any products, carefully consider your financial situation and experience level. Any opinions, news, research, analyses, prices, or other information is provided as general market commentary, and does not constitute investment advice. FXCM & TPQ will not accept liability for any loss or damage, including without limitation to, any loss of profit, which may arise directly or indirectly from use of or reliance on such information.
</font>

## Author Disclaimer

The author is neither an employee, agent nor representative of FXCM and is therefore acting independently. The opinions given are their own, constitute general market commentary, and do not constitute the opinion or advice of FXCM or any form of personal or investment advice. FXCM assumes no responsibility for any loss or damage, including but not limited to, any loss or gain arising out of the direct or indirect use of this or any other content. Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits.

## Retrieving Tick Data

In [ ]:
import time
import numpy as np
import pandas as pd
import datetime as dt
from pylab import mpl, plt

In [ ]:
plt.style.use('seaborn')
mpl.rcParams['font.family'] = 'serif'
%config InlineBackend.figure_format = 'svg'

In [ ]:
from fxcmpy import fxcmpy_tick_data_reader as tdr

In [ ]:
print(tdr.get_available_symbols())

In [ ]:
start = dt.datetime(2018, 6, 25)  
stop = dt.datetime(2018, 6, 30)  

In [ ]:
td = tdr('EURUSD', start, stop)  

In [ ]:
td.get_raw_data().info()  

In [ ]:
td.get_data().info()  

In [ ]:
td.get_data().head()

In [ ]:
sub = td.get_data(start='2018-06-29 12:00:00',
                  end='2018-06-29 12:15:00')  

In [ ]:
sub.head()

In [ ]:
sub['Mid'] = sub.mean(axis=1)  

In [ ]:
sub['SMA'] = sub['Mid'].rolling(1000).mean()  

In [ ]:
sub[['Mid', 'SMA']].plot(figsize=(10, 6), lw=0.75);
# plt.savefig('../../images/ch14/fxcm_plot_01.png')

## Retrieving Candles Data

In [ ]:
from fxcmpy import fxcmpy_candles_data_reader as cdr

In [ ]:
print(cdr.get_available_symbols())

In [ ]:
start = dt.datetime(2018, 5, 1)
stop = dt.datetime(2018, 6, 30)

`period` must be one of `m1`, `H1` or `D1`

In [ ]:
period = 'H1'  

In [ ]:
candles = cdr('EURUSD', start, stop, period)

In [ ]:
data = candles.get_data()

In [ ]:
data.info()

In [ ]:
data[data.columns[:4]].tail()  

In [ ]:
data[data.columns[4:]].tail()  

In [ ]:
data['MidClose'] = data[['BidClose', 'AskClose']].mean(axis=1)  

In [ ]:
data['SMA1'] = data['MidClose'].rolling(30).mean()  
data['SMA2'] = data['MidClose'].rolling(100).mean()  

In [ ]:
data[['MidClose', 'SMA1', 'SMA2']].plot(figsize=(10, 6));
# plt.savefig('../../images/ch14/fxcm_plot_02.png')

## Connecting to the API

In [ ]:
import fxcmpy

In [ ]:
fxcmpy.__version__

In [ ]:
api = fxcmpy.fxcmpy(config_file='../../cfg/fxcm.cfg')  

In [ ]:
instruments = api.get_instruments()

In [ ]:
print(instruments)

## Retrieving Historical Data

In [ ]:
candles = api.get_candles('USD/JPY', period='D1', number=10)  

In [ ]:
candles[candles.columns[:4]]  

In [ ]:
candles[candles.columns[4:]]  

In [ ]:
start = dt.datetime(2017, 1, 1)  
end = dt.datetime(2018, 1, 1)  

In [ ]:
candles = api.get_candles('EUR/GBP', period='D1',
                          start=start, stop=end)  

In [ ]:
candles.info()  

The parameter `period` must be one of `m1, m5, m15, m30, H1, H2, H3, H4, H6, H8, D1, W1` or `M1`.

In [ ]:
candles = api.get_candles('EUR/USD', period='m1', number=250)  

In [ ]:
candles['askclose'].plot(figsize=(10, 6))
# plt.savefig('../../images/ch14/fxcm_plot_03.png');

## Streaming Data

In [ ]:
def output(data, dataframe):
    print('%3d | %s | %s | %6.5f, %6.5f' 
          % (len(dataframe), data['Symbol'],
             pd.to_datetime(int(data['Updated']), unit='ms'), 
             data['Rates'][0], data['Rates'][1]))  

In [ ]:
api.subscribe_market_data('EUR/USD', (output,))  

In [ ]:
api.get_last_price('EUR/USD')  

In [ ]:
api.unsubscribe_market_data('EUR/USD')  

## Placing Orders

In [ ]:
api.get_open_positions()  

In [ ]:
order = api.create_market_buy_order('EUR/USD', 100)  

In [ ]:
sel = ['tradeId', 'amountK', 'currency',
       'grossPL', 'isBuy']  

In [ ]:
api.get_open_positions()[sel]  

In [ ]:
order = api.create_market_buy_order('EUR/GBP', 50)  

In [ ]:
api.get_open_positions()[sel]

In [ ]:
order = api.create_market_sell_order('EUR/USD', 25)  

In [ ]:
order = api.create_market_buy_order('EUR/GBP', 50)  

In [ ]:
api.get_open_positions()[sel]  

In [ ]:
api.close_all_for_symbol('EUR/GBP')  

In [ ]:
api.get_open_positions()[sel]

In [ ]:
api.close_all()  

In [ ]:
api.get_open_positions()

## Account Information

In [ ]:
api.get_default_account()  

In [ ]:
api.get_accounts().T  

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>